# Myanmar Programming Tutor — Before/After Evaluation

Runs inference on **50 held-out test prompts** for:
1. Base `google/gemma-3-4b-it`
2. Fine-tuned LoRA adapter from `01_train_qlora.ipynb`

Outputs:
- `results/predictions_base.jsonl`
- `results/predictions_finetuned.jsonl`
- `results/metrics_summary.json` (via `scripts/evaluate.py`)

In [ ]:
!pip install -q -r ../requirements-kaggle.txt

In [ ]:
import json
import subprocess
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ROOT = Path("/kaggle/input/myanmar-programming-tutor") if Path("/kaggle/input/myanmar-programming-tutor").exists() else Path("..")
TEST_PATH = ROOT / "data/processed/test.jsonl"
ADAPTER_PATH = Path("/kaggle/input/lora-adapter/lora_adapter") if Path("/kaggle/input/lora-adapter/lora_adapter").exists() else Path("/kaggle/working/lora_adapter")
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "google/gemma-3-4b-it"

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def build_messages(row):
    user = row["instruction"]
    if row.get("input", "").strip():
        user = f"{user}\n\n```\n{row['input']}\n```"
    return [{"role": "user", "content": user}]


test_rows = load_jsonl(TEST_PATH)
print(f"Test samples: {len(test_rows)}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def load_base_model():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )


def generate(model, messages, max_new_tokens=512):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only newly generated assistant part (best-effort split)
    if full.startswith(prompt):
        return full[len(prompt):].strip()
    return full.strip()


def run_inference(model, label):
    out_path = RESULTS_DIR / f"predictions_{label}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for row in test_rows:
            messages = build_messages(row)
            pred = generate(model, messages)
            record = {
                "id": row["id"],
                "model": label,
                "prompt": messages[0]["content"],
                "reference": row["output"],
                "prediction": pred,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {out_path}")

In [ ]:
# Base model inference
base_model = load_base_model()
run_inference(base_model, "base")
del base_model
torch.cuda.empty_cache()

In [ ]:
# Fine-tuned model inference
base_model = load_base_model()
ft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
run_inference(ft_model, "finetuned")
del ft_model, base_model
torch.cuda.empty_cache()

In [ ]:
# Compute metrics
eval_script = ROOT / "scripts/evaluate.py"
if eval_script.exists():
    subprocess.run([
        "python", str(eval_script),
        "--base", str(RESULTS_DIR / "predictions_base.jsonl"),
        "--finetuned", str(RESULTS_DIR / "predictions_finetuned.jsonl"),
        "--output", str(RESULTS_DIR / "metrics_summary.json"),
    ], check=True)
    print(open(RESULTS_DIR / "metrics_summary.json").read())
else:
    print("Copy scripts/evaluate.py into Kaggle dataset to auto-compute metrics.")